# Step 2.1 - EV charging systems

Copia del notebook allegato, aggiornata con le modifiche del PDF del 20 novembre 2026.
Le funzioni e le analisi originali sono mantenute. Modifiche: selezione per famiglia,
alias esatti verificabili, CPC coerenti in Q3, soglie nella tabella RTA e legenda corretta.

**Prima esecuzione:** su TIP scegli Restart Kernel and Run All Cells. Il default è TEST;
usa PROD soltanto per i risultati finali dopo i controlli. I test SQL aggiuntivi sono
disattivati: abilita RUN_SQL_CHECKS nella configurazione per eseguirli.

**SDG:** la sezione facoltativa si abilita con RUN_SDG_ANALYSIS. I CPC suggeriscono
solo candidati esplorativi, non etichette ufficiali. Le tabelle esplorative sono automatiche; i conteggi confermati richiedono
sdg_review.csv compilato e revisionato. Nessun mapping o dato SDG è inventato.

Gli output salvati sono stati rimossi per evitare di confonderli con una nuova esecuzione.
Questa versione non è stata eseguita integralmente su PATSTAT; è stata verificata offline.


# Setup and configuration

In [ ]:
import re
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from scipy import stats # linear regression (Lotka) and Kendall's tau
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons: standardised name, sector, country
    TLS207_PERS_APPLN,          # person <-> application links (applicant / inventor)
    TLS209_APPLN_IPC,           # IPC classification
    TLS211_PAT_PUBLN,           # publications: grants and their dates
    TLS224_APPLN_CPC,           # CPC classification
)

# 1) Environment
ENV = "TEST" # Use "TEST" while debugging, switch to "PROD" for final deck numbers

# 2) Applicant names (Section 2)
NAME_METHOD = "custom" # "cleanco" or "dictionary" or "custom"?

# 3) Our sector: EV Charging systems (Section 5)
SECTOR_NAME = "EV charging systems"

SECTOR_CPC_PREFIXES = (
    "B60L53",       # Methods of charging batteries, specially adapted for electric vehicles; charging stations
    "H02J7/00",     # Circuit arrangements for charging or depolarising batteries (often used alongside B60L53)
    "Y02T10/70",    # Energy storage systems for electromobility / charging (climate change mitigation tagging)
    "Y02T90/12",    # Electric vehicle charging stations (specifically infrastructure)
    "Y02T90/14",    # Plug-in electric vehicles / charging interfaces
)

# Optional: Add keywords to capture patents not yet tagged with B60L53, or leave empty () to rely strictly on CPC
SECTOR_KEYWORDS = (
    "electric vehicle charging", 
    "ev charging", 
    "charging station", 
    "wireless power transfer electric vehicle",
)

YEAR_MIN, YEAR_MAX = 2000, 2025
LAST_COMPLETE_YEAR = YEAR_MAX - 3    # Keeps data up to 2022 (avoids incomplete recent filing years)
IPR_TYPE = "PI"                      # Patents of invention only
SPLIT_YEAR = 2015                    # Good split year for EVs (around when fast-charging & mass adoption ramped up)
WINDOW = 3                           # 3-year rolling/consecutive windows for concentration trends
TOP_N = 15                           # Number of top applicants to display

COUNTRY_NAMES = {
    "AU": "Australia", "CA": "Canada", "FR": "France", "DE": "Germany",
    "IL": "Israel", "IT": "Italy", "JP": "Japan", "NL": "Netherlands",
    "SE": "Sweden", "CH": "Switzerland", "GB": "UK", "US": "USA",
}

# Results stay in this notebook. Turn on only if CSV/PNG exports are wanted.
EXPORT_FILES = False
PAST_PERIOD = (2013, 2017)
RECENT_PERIOD = (2018, 2022)
MIN_GROWTH_BASE = 10

# Step 2.1: parametri aggiuntivi, senza cambiare la ricerca settoriale originale.
SUBFIELD_CPC_PREFIXES = ("B60L53/", "H02J7/", "Y02T90/", "Y02T10/70")
MIN_RTA_FAMILIES = 5
MIN_RTA_PORTFOLIO = 10
RUN_SQL_CHECKS = False
RUN_SDG_ANALYSIS = True

# Analisi SDG automatica esplorativa: tutte le domande osservate, a blocchi.
SDG_BATCH_SIZE = 500
SDG_MAX_APPLN = None  # Per una prima prova puoi impostare 200.


1) Set output directory and initialize the PATSTAT connection 

In [ ]:
# Output directory
OUT_DIR = Path("outputs/step2")
if EXPORT_FILES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

patstat = PatstatClient(env=ENV)
db = patstat.orm()
print(f"Connected to PATSTAT ({ENV}) | output dir: {OUT_DIR.resolve()}")


In [ ]:
def save_table(df: pd.DataFrame, name: str, index: bool = True) -> Path | None:
    if not EXPORT_FILES:
        return None
    path = OUT_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print(f"saved table -> {path}")
    return path

def save_fig(fig: plt.Figure, name: str) -> Path | None:
    if not EXPORT_FILES:
        return None
    path = OUT_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved figure -> {path}")
    return path

def lorenz_curve(counts) -> tuple[np.ndarray, np.ndarray]:
    """Return (cum_firm_share, cum_patent_share) starting at (0, 0) and ending at (1, 1)."""
    c = np.sort(np.asarray(counts, dtype=float))
    c = c[c > 0]
    n = len(c)
    if n == 0:
        return np.array([0.0, 1.0]), np.array([0.0, 1.0])
    cum_firms = np.concatenate(([0.0], np.arange(1, n + 1) / n))
    cum_patents = np.concatenate(([0.0], np.cumsum(c) / c.sum()))
    return cum_firms, cum_patents

def gini_from_counts(counts) -> float:
    if len(counts) == 0 or np.sum(counts) <= 0:
        return np.nan
    x, y = lorenz_curve(counts)
    return float(1.0 - 2.0 * np.trapezoid(y, x))

def concentration_measures(counts, label: str = "") -> dict:
    """Concentration of full attributions or fractional credits, as supplied."""
    c = np.sort(np.asarray(counts, dtype=float))[::-1]
    c = c[np.isfinite(c) & (c > 0)]
    total = float(c.sum())
    shares = c / total if total > 0 else np.array([])
    hh = float(np.sum(shares ** 2)) if total > 0 else np.nan
    out = {
        "group": label,
        "applicants": len(c),
        "attributions": total,
        "CR4": float(shares[:4].sum()) if total > 0 else np.nan,
        "CR8": float(shares[:8].sum()) if total > 0 else np.nan,
        "HH": hh,
        "1/HH": 1.0 / hh if hh > 0 else np.nan,
        "Gini": gini_from_counts(c),
        "Lotka L": np.nan,
    }
    # Lotka concerns integer portfolio sizes, not rounded fractional credits.
    if len(c) > 0 and np.allclose(c, np.round(c)):
        sizes, freqs = np.unique(c.astype(int), return_counts=True)
        mask = sizes >= 2
        if mask.sum() >= 3:
            slope, _, _, _, _ = stats.linregress(np.log10(sizes[mask]), np.log10(freqs[mask]))
            out["Lotka L"] = float(-slope)
    return out


def build_sector_query(
    cpc_prefixes: tuple[str, ...] = (),
    keywords: tuple[str, ...] = (),
    year_min: int = YEAR_MIN,
    year_max: int = LAST_COMPLETE_YEAR,
    ipr_type: str = IPR_TYPE,
):
    """Query patent applications matching CPC prefixes and/or title/abstract keywords."""
    q = (
        db.query(
            TLS201_APPLN.appln_id,
            TLS201_APPLN.docdb_family_id,
            TLS201_APPLN.appln_filing_year,
            TLS201_APPLN.earliest_filing_year,
            TLS201_APPLN.appln_auth,
            TLS206_PERSON.person_id,
            TLS206_PERSON.person_name,
            TLS206_PERSON.psn_name,
            TLS206_PERSON.psn_sector,
            TLS206_PERSON.person_ctry_code,
        )
        .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
        .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
        .filter(
            TLS207_PERS_APPLN.applt_seq_nr > 0,
            TLS201_APPLN.ipr_type == ipr_type,
            TLS201_APPLN.earliest_filing_year.between(year_min, year_max),
        )
    )

    conditions = []
    if cpc_prefixes:
        cpc_subq = (
            db.query(TLS224_APPLN_CPC.appln_id)
            .filter(or_(*[func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", "").like(p.replace(" ", "") + "%") for p in cpc_prefixes]))
        )
        conditions.append(TLS201_APPLN.appln_id.in_(cpc_subq))

    if keywords:
        kw_conds = []
        for kw in keywords:
            pat = f"%{kw}%"
            kw_conds.append(
                TLS201_APPLN.appln_id.in_(
                    db.query(TLS202_APPLN_TITLE.appln_id).filter(TLS202_APPLN_TITLE.appln_title.ilike(pat))
                )
            )
            kw_conds.append(
                TLS201_APPLN.appln_id.in_(
                    db.query(TLS203_APPLN_ABSTR.appln_id).filter(TLS203_APPLN_ABSTR.appln_abstract.ilike(pat))
                )
            )
        conditions.append(or_(*kw_conds))

    if conditions:
        selected_families = db.query(
            TLS201_APPLN.docdb_family_id
        ).filter(
            or_(*conditions),
            TLS201_APPLN.ipr_type == ipr_type,
            TLS201_APPLN.earliest_filing_year.between(year_min, year_max),
            TLS201_APPLN.docdb_family_id > 0,
        ).distinct()
        q = q.filter(
            TLS201_APPLN.docdb_family_id.in_(selected_families)
        )

    return q.distinct()


# Applicants' names cleaning
1) Load Gold-Standard Benchmark and basic cleaning

Loads the manually labeled benchmark file containing known duplicate company names (comp) and their true company group IDs (Group, then applies a basic cleanup (converting to uppercase, removing punctuation, and fixing extra spaces) so we have a baseline for comparison

## Prerequisiti del benchmark

Il CSV del tutorial 1 deve essere accessibile dal notebook. Se l'import di cleanco
fallisce, esegui una volta `%pip install cleanco` in una cella, riavvia il kernel e
riesegui tutto. Il notebook non installa automaticamente pacchetti.


In [ ]:
from cleanco import basename

gold_candidates = [
    Path("PATSTAT_500_label_datav4.csv"),
    Path("tutorial_1/PATSTAT_500_label_datav4.csv"),
    Path("innovation-management-datascience26-main/tutorial_1/PATSTAT_500_label_datav4.csv"),
    Path("innovation-management-datascience26/tutorial_1/PATSTAT_500_label_datav4.csv"),
]
GOLD_STANDARD_FILE = next((p for p in gold_candidates if p.is_file()), None)
if GOLD_STANDARD_FILE is None:
    raise FileNotFoundError("Carica PATSTAT_500_label_datav4.csv accanto al notebook")
gold_df = pd.read_csv(GOLD_STANDARD_FILE)
print(f"Loaded {len(gold_df)} rows | unique raw names: {gold_df['comp'].nunique()} | gold clusters: {gold_df['Group'].nunique()}")

def basic_clean(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return ""
    s = name.upper()
    s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

gold_df["basic"] = gold_df["comp"].apply(basic_clean)


2) Cleanco and Dictionary cleaning methods

Defines two standard ways to strip legal entity suffixes (like Inc., Ltd) from company names, first using the cleanco python library, and second using a fixed list of common international legal words. Both techniques are applied to the benchmark dataset.

In [ ]:
def clean_with_cleanco(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return ""
    s = basic_clean(name)
    prev = None
    while prev != s:
        prev = s
        s = basename(s).strip()
    return s

LEGAL_SUFFIXES = [
    "LIMITED", "LTD", "INCORPORATED", "INC", "CORPORATION", "CORP",
    "COMPANY", "CO", "LLC", "LLP", "LP", "PLC", "AG", "GMBH", "SA", "SAS",
    "SARL", "SPA", "SRL", "BV", "NV", "AB", "OY", "AS", "ASA", "KK",
    "KABUSHIKI KAISHA", "KG", "OHG", "SE", "PTE", "PTY", "BHD", "SDN"
]
_SUFFIX_PATTERN = re.compile(r"\b(?:" + "|".join(LEGAL_SUFFIXES) + r")\b")

def clean_with_dictionary(name: str) -> str:
    s = basic_clean(name)
    s = _SUFFIX_PATTERN.sub("", s)
    return re.sub(r"\s+", " ", s).strip()

gold_df["cleanco"] = gold_df["comp"].apply(clean_with_cleanco)
gold_df["dictionary"] = gold_df["comp"].apply(clean_with_dictionary)


3) Pairwise evaluation function and baseline scores

Checks every pair of company names to see how accurately each cleaning method groups the subsidiaries together without accidentally merging unrelated companies, then outputs a table of Precision, Recall, and F1 scores.

In [ ]:
from itertools import combinations

def evaluate_harmonisation(df: pd.DataFrame, pred_col: str, true_col: str = "Group") -> dict:
    """Compute pairwise Precision, Recall, and F1 over all pairs of rows."""
    n = len(df)
    preds = df[pred_col].fillna("").values
    trues = df[true_col].values

    tp = fp = fn = 0
    for i, j in combinations(range(n), 2):
        same_pred = (preds[i] != "") and (preds[i] == preds[j])
        same_true = (trues[i] == trues[j])
        if same_pred and same_true:
            tp += 1
        elif same_pred and not same_true:
            fp += 1
        elif not same_pred and same_true:
            fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "clusters": int(pd.Series(preds[preds != ""]).nunique()),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "F1": round(f1, 4),
    }

baseline_results = pd.DataFrame([
    {"method": "raw (comp)", **evaluate_harmonisation(gold_df, "comp")},
    {"method": "basic", **evaluate_harmonisation(gold_df, "basic")},
    {"method": "cleanco", **evaluate_harmonisation(gold_df, "cleanco")},
    {"method": "dictionary", **evaluate_harmonisation(gold_df, "dictionary")},
])
display(baseline_results)



4) Custom harmonization function

Builds on cleanco by removing generic business and geographic words (such as GROUP, HOLDINGS, SYSTEMS, DEUTSCHLAND, CHINA) that frequently split a single multinational company into multiple entries, then tests whether this improves the F1 score.

In [ ]:
def my_harmonisation(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return ""
    s = clean_with_cleanco(name)
    s = re.sub(r"[^\w\s]", " ", s)
    noise_words = [
        "CORPORATION", "COMPANY", "GROUP", "HOLDINGS", "HOLDING",
        "INTERNATIONAL", "TECHNOLOGIES", "TECHNOLOGY", "SYSTEMS",
        "INDUSTRIES", "ENTERPRISES", "KABUSHIKI", "KAISHA", "KK",
        "DEUTSCHLAND", "AMERICA", "EUROPE", "JAPAN", "CHINA"
    ]
    for word in noise_words:
        s = re.sub(rf"\b{word}\b", "", s)
    return re.sub(r"\s+", " ", s).strip()

gold_df["my_clean"] = gold_df["comp"].apply(my_harmonisation)
scores_my = evaluate_harmonisation(gold_df, "my_clean", "Group")
print("My harmonisation scores:", scores_my)


5) Function to select a method

Creates a single helper function (consolidated_names) that takes raw PATSTAT applicant columns and applies whichever cleaning strategy (psn, cleanco, dictionary, or custom) you configured at the top of the notebook.

In [ ]:
def consolidated_names(df: pd.DataFrame, method: str = NAME_METHOD) -> pd.Series:
    """Return a Series of harmonised applicant names according to the selected method."""
    raw = df["person_name"].fillna("")
    if method == "psn":
        psn = df["psn_name"].fillna("").astype(str).str.strip()
        return psn.where(psn.ne(""), raw).apply(basic_clean)
    elif method == "cleanco":
        return raw.apply(clean_with_cleanco)
    elif method == "dictionary":
        return raw.apply(clean_with_dictionary)
    elif method == "custom":
        return raw.apply(my_harmonisation)
    else:
        raise ValueError(f"Unknown NAME_METHOD: {method}")


# EV Charging Systems analysis

La selezione ora individua prima le famiglie che corrispondono ai criteri originari.
I richiedenti sono recuperati dai membri osservati con i filtri originali anno/tipo PI.
Gli alias manuali sono esatti e inizialmente vuoti: controllare i nomi prima di compilarli.
I richiedenti osservati non equivalgono ai proprietari attuali o a gruppi societari verificati.


In [ ]:
# Query PATSTAT for EV charging patents and apply name harmonisation + major-player overrides
raw_sector = pd.read_sql(build_sector_query(SECTOR_CPC_PREFIXES, SECTOR_KEYWORDS).statement, db.bind)
raw_sector["clean_name"] = consolidated_names(raw_sector, method=NAME_METHOD)
raw_sector = raw_sector[raw_sector["clean_name"] != ""].copy()

# Merge common corporate variants in EV charging
EV_OVERRIDES = {
    "STATE GRID": "STATE GRID CORP OF CHINA",
    "TOYOTA": "TOYOTA MOTOR",
    "ABB ": "ABB",
    "SIEMENS": "SIEMENS",
    "ROBERT BOSCH": "BOSCH",
    "BYD": "BYD",
    "HYUNDAI MOTOR": "HYUNDAI MOTOR",
    "HONDA MOTOR": "HONDA MOTOR",
    "FORD GLOBAL": "FORD",
    "GM GLOBAL": "GENERAL MOTORS",
    "KIA": "KIA",
}
# EV_OVERRIDES sopra resta come riferimento delle vecchie regole per prefisso.
# Non viene piu' applicato. Inserire qui SOLO alias esatti verificati a mano.
VERIFIED_ALIASES = {}
raw_sector["name_before_override"] = raw_sector["clean_name"]
raw_sector["clean_name"] = raw_sector["clean_name"].replace(VERIFIED_ALIASES)
display(raw_sector[["person_name", "psn_name", "name_before_override", "clean_name"]]
        .drop_duplicates().head(100))

print(f"{SECTOR_NAME} ({ENV}): {len(raw_sector):,} rows | {raw_sector['docdb_family_id'].nunique():,} unique families | {raw_sector['clean_name'].nunique():,} unique applicants")


# One observed family-applicant pair, after the existing name cleaning and overrides.
# The year is the minimum observed in this extraction, not a new query of all family members.
family_years = (raw_sector.groupby("docdb_family_id")["earliest_filing_year"]
               .min().rename("earliest_filing_year").reset_index())
family_applicants = raw_sector[["docdb_family_id", "clean_name"]].drop_duplicates()
family_applicants = family_applicants.merge(family_years, on="docdb_family_id", validate="many_to_one")
family_applicants["n_applicants"] = family_applicants.groupby("docdb_family_id")["clean_name"].transform("nunique")
family_applicants["fractional_credit"] = 1 / family_applicants["n_applicants"]

print("Unique observed families:", family_applicants.docdb_family_id.nunique())
print("Full family-applicant attributions:", len(family_applicants))
print("Fractional family credits:", round(family_applicants.fractional_credit.sum(), 2))


## Step 2.1 - controlli di coerenza

Questi controlli verificano la struttura dei dati osservati, non la completezza di PATSTAT.
Gli assert devono passare prima di interpretare le analisi successive.


In [ ]:
# Nuova cella dopo la cella 18: invarianti dei conteggi.
assert not family_applicants.duplicated(
    ['docdb_family_id', 'clean_name']).any()
credits = family_applicants.groupby('docdb_family_id')
assert np.allclose(credits.fractional_credit.sum(), 1)
assert family_applicants.earliest_filing_year.between(
    YEAR_MIN, LAST_COMPLETE_YEAR).all()

# Cella 22, riga 82: cambiare soltanto la stringa label in:
# 'Assenti ora fra gli attivi nella finestra precedente (%)'


print("Controlli di coerenza superati")


2) Question 1: How concentrated is the field, and is it consolidating?

Compare full and fractional counting using the same observed family-applicant pairs. `unique_families` counts inventions; `attributions` sums the counting credits. Full attributions can exceed unique families. Trends use complete three-year windows only: with cutoff 2022, 2021–2022 is excluded. Concentration describes patenting activity, not market shares.

In [ ]:
# Q1: compare full and fractional counting on the SAME observed families.
fam_per_app = family_applicants.groupby("clean_name")["docdb_family_id"].nunique()
fractional_per_app = family_applicants.groupby("clean_name")["fractional_credit"].sum()
overall_conc = pd.DataFrame([
    concentration_measures(fam_per_app, label="Full family-applicant attributions"),
    concentration_measures(fractional_per_app, label="Fractional family credits"),
])
overall_conc["unique_families"] = family_applicants.docdb_family_id.nunique()
display(overall_conc)
save_table(overall_conc, "q1_overall_concentration", index=False)

# Complete, consecutive three-year windows. The partial final window is excluded.
trend_rows = []
for w_start in range(YEAR_MIN, LAST_COMPLETE_YEAR - WINDOW + 2, WINDOW):
    w_end = w_start + WINDOW - 1
    sub = family_applicants[family_applicants.earliest_filing_year.between(w_start, w_end)]
    counts = sub.groupby("clean_name")["docdb_family_id"].nunique()
    fractional = sub.groupby("clean_name")["fractional_credit"].sum()
    for method, values in [("full", counts), ("fractional", fractional)]:
        row = concentration_measures(values, label=f"{w_start}-{w_end}")
        row["counting"] = method
        row["unique_families"] = sub.docdb_family_id.nunique()
        trend_rows.append(row)
conc_trends = pd.DataFrame(trend_rows)
display(conc_trends)
save_table(conc_trends, "q1_concentration_trends", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for method in ["full", "fractional"]:
    line = conc_trends[conc_trends.counting == method]
    axes[0].plot(line["group"], line.CR4 * 100, marker="o", label=method)
    axes[1].plot(line["group"], line["1/HH"], marker="o", label=method)
axes[0].set_title("Concentration: CR4")
axes[0].set_ylabel("Share of the selected counting total (%)")
axes[1].set_title("Effective number of applicants (1/HHI)")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
fig.tight_layout()
save_fig(fig, "q1_concentration_trends")
plt.show()


3) Question 2: Who are the incumbents? And who are the new entrants?

Ranks the top 15 patent applicants in EV charging systems, labels each as an incumbent (active before SPLIT_YEAR = 2015) or entrant (first active in or after 2015), and computes window-by-window entry and disappearance rates (compute_entry_exit)

“Entrant” is a first observation in this dataset, not necessarily a newly founded company or a first-ever patent applicant. The first window is left censored. Disappearance means no observed family in the next complete window; it does not establish business exit. Applicant types can include companies, individuals and universities.

In [ ]:
# Define cohort functions and run Q2: Top 15 applicants (incumbent vs. entrant) and 3-year entry/exit dynamics
def classify_incumbents_entrants(
    df: pd.DataFrame,
    split_year: int = SPLIT_YEAR,
    applicant_col: str = "clean_name",
    year_col: str = "earliest_filing_year",
    unit_col: str = "docdb_family_id",
) -> pd.DataFrame:
    """Classify applicants as 'incumbent' (active < split_year) or 'entrant' (first active >= split_year)."""
    first_year = df.groupby(applicant_col)[year_col].min().rename("first_year")
    last_year = df.groupby(applicant_col)[year_col].max().rename("last_year")
    total_fams = df.groupby(applicant_col)[unit_col].nunique().rename("total_families")
    pre_fams = (
        df[df[year_col] < split_year]
        .groupby(applicant_col)[unit_col]
        .nunique()
        .rename("pre_split_families")
    )
    post_fams = (
        df[df[year_col] >= split_year]
        .groupby(applicant_col)[unit_col]
        .nunique()
        .rename("post_split_families")
    )

    summary = pd.concat([first_year, last_year, total_fams, pre_fams, post_fams], axis=1).fillna(0)
    summary["pre_split_families"] = summary["pre_split_families"].astype(int)
    summary["post_split_families"] = summary["post_split_families"].astype(int)
    summary["status"] = np.where(summary["first_year"] < split_year, "incumbent", "entrant")
    return summary.sort_values("total_families", ascending=False)

def compute_entry_exit(
    df: pd.DataFrame,
    window: int = WINDOW,
    year_min: int = YEAR_MIN,
    year_max: int = LAST_COMPLETE_YEAR,
    applicant_col: str = "clean_name",
    unit_col: str = "docdb_family_id",
) -> pd.DataFrame:
    """First observations and absence in the next complete window, not business exits."""
    rows = []
    seen_so_far = set()
    prev_active = set()
    for w_start in range(year_min, year_max - window + 2, window):
        w_end = w_start + window - 1
        sub = df[df.earliest_filing_year.between(w_start, w_end)]
        active = set(sub[applicant_col].dropna().unique())
        new_to_sector = active - seen_so_far
        disappeared = prev_active - active
        total_fams = sub[unit_col].nunique()
        entrant_fams = sub[sub[applicant_col].isin(new_to_sector)][unit_col].nunique()
        first_window = w_start == year_min
        rows.append({
            "window": f"{w_start}-{w_end}",
            "active_applicants": len(active),
            "new_to_sector": np.nan if first_window else len(new_to_sector),
            "entrants_share_families_%": (100 * entrant_fams / total_fams
                if total_fams > 0 and not first_window else np.nan),
            "disappeared_from_prev": np.nan if first_window else len(disappeared),
            "rate_of_disappearance_%": (100 * len(disappeared) / len(prev_active)
                if prev_active else np.nan),
            "left_censored": first_window,
        })
        seen_so_far.update(active)
        prev_active = active
    return pd.DataFrame(rows)


# Run Q2 analysis
app_summary = classify_incumbents_entrants(family_applicants, split_year=SPLIT_YEAR)
top_applicants = app_summary.head(TOP_N)
display(top_applicants)
save_table(top_applicants, "q2_top_applicants")

entry_exit_df = compute_entry_exit(family_applicants, window=WINDOW)
display(entry_exit_df)
save_table(entry_exit_df, "q2_entry_exit", index=False)

# Plot entry vs. disappearance dynamics
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(entry_exit_df["window"], entry_exit_df["entrants_share_families_%"], alpha=0.7, label="Entrants' share of families (%)")
ax.plot(entry_exit_df["window"], entry_exit_df["rate_of_disappearance_%"], marker="o", color="tab:red", label="Assenti ora fra gli attivi nella finestra precedente (%)")
ax.set_ylabel("Percentage (%)")
ax.set_title(f"{SECTOR_NAME}: Entrant Patent Share & Applicant Inactivity Between Windows")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
save_fig(fig, "q2_entry_exit")
plt.show()


4) Question 3: Which subfields are growing faster?

The extraction still uses TLS224_APPLN_CPC and the application IDs already in `raw_sector`. CPC symbols are preserved in full. Growth compares 2013–2017 and 2018–2022, both five years. Show size and growth separately; a zero starting base has no finite growth rate. Classifications overlap, and these counts describe the selected applications, not all classifications of every worldwide family member.
Step 2.1: Q3 include anche Y02T10/70 e discendenti, prima esclusi dal filtro dei sotto-settori. Le classificazioni restano sovrapposte.


In [ ]:
# Q3: same PATSTAT table and same application IDs as in the original notebook.
appln_ids = raw_sector["appln_id"].unique().tolist()
cpc_rows = []
chunk_size = 5000
for i in range(0, len(appln_ids), chunk_size):
    chunk = appln_ids[i:i + chunk_size]
    q_cpc = db.query(TLS224_APPLN_CPC.appln_id, TLS224_APPLN_CPC.cpc_class_symbol).filter(
        TLS224_APPLN_CPC.appln_id.in_(chunk)
    )
    cpc_rows.append(pd.read_sql(q_cpc.statement, db.bind))

if cpc_rows:
    cpc_df = pd.concat(cpc_rows, ignore_index=True)
else:
    cpc_df = pd.DataFrame(columns=["appln_id", "cpc_class_symbol"])

# Keep the FULL symbol: a fixed character slice can merge different CPC subgroups.
cpc_df["cpc_subgroup"] = cpc_df.cpc_class_symbol.fillna("").str.replace(" ", "", regex=False)
relevant_cpc = cpc_df[
    cpc_df.cpc_subgroup.str.startswith(SUBFIELD_CPC_PREFIXES)
].copy()
appln_families = raw_sector[["appln_id", "docdb_family_id"]].drop_duplicates()
merged_cpc = relevant_cpc.merge(appln_families, on="appln_id", validate="many_to_one")
merged_cpc = merged_cpc.merge(family_years, on="docdb_family_id", validate="many_to_one")
merged_cpc = merged_cpc.drop_duplicates(["docdb_family_id", "cpc_subgroup"])

def subfield_growth_table(df, past=(2013, 2017), recent=(2018, 2022), min_base=10):
    """Compare annual family frequencies; no artificial +1 for a zero base."""
    if past[1] < past[0] or recent[1] < recent[0] or past[1] >= recent[0]:
        raise ValueError("Choose two ordered, non-overlapping periods.")
    data = df.drop_duplicates(["docdb_family_id", "cpc_subgroup"])
    pre = data[data.earliest_filing_year.between(*past)].groupby("cpc_subgroup").docdb_family_id.nunique()
    post = data[data.earliest_filing_year.between(*recent)].groupby("cpc_subgroup").docdb_family_id.nunique()
    result = pd.concat([pre.rename("families_past"), post.rename("families_recent")], axis=1).fillna(0)
    result[["families_past", "families_recent"]] = result[["families_past", "families_recent"]].astype(int)
    result["annual_past"] = result.families_past / (past[1] - past[0] + 1)
    result["annual_recent"] = result.families_recent / (recent[1] - recent[0] + 1)
    result["growth_%"] = np.nan
    positive_base = result.annual_past > 0
    result.loc[positive_base, "growth_%"] = 100 * (
        result.loc[positive_base, "annual_recent"] / result.loc[positive_base, "annual_past"] - 1)
    result["growth_status"] = "comparable_base"
    result.loc[result.families_past < min_base, "growth_status"] = "small_base"
    result.loc[result.families_past == 0, "growth_status"] = "new_in_recent_period"
    return result.reset_index().sort_values("families_recent", ascending=False).reset_index(drop=True)

assert YEAR_MIN <= PAST_PERIOD[0] <= PAST_PERIOD[1] < RECENT_PERIOD[0] <= RECENT_PERIOD[1] <= LAST_COMPLETE_YEAR
subfield_growth = subfield_growth_table(merged_cpc, PAST_PERIOD, RECENT_PERIOD, MIN_GROWTH_BASE)
print("Largest recent subfields:", RECENT_PERIOD)
display(subfield_growth.head(15))
print("Fastest growth with a sufficient starting base:")
display(subfield_growth[subfield_growth.growth_status == "comparable_base"].sort_values("growth_%", ascending=False).head(15))
save_table(subfield_growth, "q3_subfield_growth", index=False)


5) Is concentration sensitive to co-applicants?

The Lorenz curves and ranking compare full attributions and fractional credits. Applicant names come directly from the existing cleaning and override rules. The same rules apply to both calculations; neither is a new corporate-ownership database.

In [ ]:
ranking_comparison = pd.concat([
    fam_per_app.rename("full_families"),
    fractional_per_app.rename("fractional_families"),
], axis=1)
ranking_comparison["rank_full"] = ranking_comparison.full_families.rank(ascending=False, method="min")
ranking_comparison["rank_fractional"] = ranking_comparison.fractional_families.rank(ascending=False, method="min")
ranking_comparison["rank_change"] = ranking_comparison.rank_full - ranking_comparison.rank_fractional
display(ranking_comparison.sort_values("fractional_families", ascending=False).head(TOP_N))

fig, ax = plt.subplots(figsize=(6, 5))
for label, values in [("Full attributions", fam_per_app), ("Fractional credits", fractional_per_app)]:
    x, y = lorenz_curve(values)
    ax.plot(x, y, label=label)
ax.plot([0, 1], [0, 1], "--", color="grey", label="Equal distribution")
ax.set_xlabel("Cumulative share of observed applicants")
ax.set_ylabel("Cumulative share of attributions / credits")
ax.set_title("EV charging: Lorenz curves")
ax.legend()
fig.tight_layout()
plt.show()

6) Inactivity with three complete years of follow-up

For each year t, consider applicants active in t. An applicant is inactive if no new family is observed in t+1, t+2 and t+3. A later return is allowed. With the current cutoff, t=2019 is the last measurable cohort. More recent cohorts stay missing, not zero.

In [ ]:
def compute_inactivity(df, year_min=YEAR_MIN, year_max=LAST_COMPLETE_YEAR):
    rows = []
    data = df[["clean_name", "docdb_family_id", "earliest_filing_year"]].drop_duplicates()
    for year in range(year_min, year_max + 1):
        active = set(data.loc[data.earliest_filing_year == year, "clean_name"])
        inactive_n = np.nan
        rate = np.nan
        complete = year + 3 <= year_max
        if complete:
            following = data[data.earliest_filing_year.between(year + 1, year + 3)]
            inactive_n = len(active - set(following.clean_name))
            if active:
                rate = 100 * inactive_n / len(active)
        rows.append({"year": year, "active_applicants": len(active),
                     "inactive_3y": inactive_n, "inactivity_rate_%": rate,
                     "complete_followup": complete})
    return pd.DataFrame(rows)

inactivity_3y = compute_inactivity(family_applicants)
display(inactivity_3y)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(inactivity_3y.year, inactivity_3y["inactivity_rate_%"], marker="o")
ax.set_xlabel("Initial year t")
ax.set_ylabel("Applicants inactive in t+1, t+2, t+3 (%)")
ax.set_title("Three-year patenting inactivity — complete follow-up only")
fig.tight_layout()
plt.show()

7) Which applicants specialize in which charging technologies?

RTA compares the share of an applicant's CPC-coded families in a subgroup with the corresponding share in the whole CPC-coded sector. RTA > 1 means relative specialization, not better quality. A family can appear in more than one subgroup. Denominators include all observed families with at least one relevant CPC, even if the plot shows only the largest applicants. Check the underlying counts before interpreting high ratios on small portfolios.

In [ ]:
def compute_specialization(applicants, cpc):
    names = applicants[["docdb_family_id", "clean_name"]].drop_duplicates()
    codes = cpc[["docdb_family_id", "cpc_subgroup"]].drop_duplicates()
    pairs = names.merge(codes, on="docdb_family_id", how="inner")
    columns = ["clean_name", "cpc_subgroup", "families_in_subfield", "coded_portfolio",
               "sector_subfield_families", "sector_coded_families", "RTA"]
    if pairs.empty:
        return pd.DataFrame(columns=columns)
    portfolio = pairs.groupby("clean_name").docdb_family_id.nunique().rename("coded_portfolio")
    counts = pairs.groupby(["clean_name", "cpc_subgroup"]).docdb_family_id.nunique().rename("families_in_subfield")
    sector_counts = codes.groupby("cpc_subgroup").docdb_family_id.nunique().rename("sector_subfield_families")
    result = counts.reset_index().merge(portfolio, on="clean_name", validate="many_to_one")
    result = result.merge(sector_counts, on="cpc_subgroup", validate="many_to_one")
    result["sector_coded_families"] = codes.docdb_family_id.nunique()
    applicant_share = result.families_in_subfield / result.coded_portfolio
    sector_share = result.sector_subfield_families / result.sector_coded_families
    result["RTA"] = applicant_share / sector_share
    return result[columns].sort_values(["clean_name", "cpc_subgroup"]).reset_index(drop=True)

specialization = compute_specialization(family_applicants, merged_cpc)
display(specialization[specialization.families_in_subfield >= 5].sort_values("RTA", ascending=False).head(25))

if not specialization.empty:
    selected_applicants = fractional_per_app.nlargest(12).index
    selected_cpc = merged_cpc.groupby("cpc_subgroup").docdb_family_id.nunique().nlargest(12).index
    matrix = specialization.pivot(index="clean_name", columns="cpc_subgroup", values="RTA")
    matrix = matrix.reindex(index=selected_applicants, columns=selected_cpc)
    # Zero applies only to a coded portfolio with no family in that subgroup.
    has_coded_portfolio = matrix.index.isin(specialization.clean_name)
    matrix.loc[has_coded_portfolio] = matrix.loc[has_coded_portfolio].fillna(0)
    fig, ax = plt.subplots(figsize=(12, 7))
    image = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap="YlGnBu", vmin=0)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=60, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_title("Relative technological specialization (RTA)")
    fig.colorbar(image, ax=ax, label="RTA (1 = sector incidence)")
    fig.tight_layout()
    plt.show()
else:
    print("No relevant CPC available: specialization cannot be calculated.")

8) Read the results together

This table brings together the existing applicant classification, portfolio size, recent activity and the strongest observed specialization. It supports discussion of a shortlist; it is not an automatic recommendation. Verify applicant type and corporate ownership separately before calling an actor a company or an acquisition target. Any Toyota client benchmark requires manually verified name aliases.
Step 2.1: il massimo RTA nella tabella finale richiede almeno MIN_RTA_FAMILIES nel sotto-settore e MIN_RTA_PORTFOLIO nel portafoglio codificato. Gli alias Toyota non sono assunti verificati: controllarli manualmente.


In [ ]:
recent_names = family_applicants[family_applicants.earliest_filing_year.between(*RECENT_PERIOD)]
recent_volume = recent_names.groupby("clean_name").docdb_family_id.nunique().rename("recent_families")
discussion_table = app_summary.join(ranking_comparison[["fractional_families", "rank_change"]])
discussion_table = discussion_table.join(recent_volume).fillna({"recent_families": 0})
eligible = specialization[
    specialization.families_in_subfield.ge(MIN_RTA_FAMILIES)
    & specialization.coded_portfolio.ge(MIN_RTA_PORTFOLIO)
]
if not eligible.empty:
    strongest = eligible.loc[
        eligible.groupby('clean_name').RTA.idxmax()
    ].set_index('clean_name')
    strongest = strongest[['cpc_subgroup','RTA','families_in_subfield']]
    strongest = strongest.rename(columns={
        'cpc_subgroup': 'strongest_CPC',
        'RTA': 'strongest_RTA',
        'families_in_subfield': 'specialization_base'})
    discussion_table = discussion_table.join(strongest)
display(discussion_table.sort_values("total_families", ascending=False).head(TOP_N))

# Controlli SQL facoltativi su PATSTAT

Abilita **RUN_SQL_CHECKS=True** nella configurazione e riesegui dall'inizio.
Le query sono quelle del PDF: usano standard SQL e un massimo di 200 ID osservati.
Il campione ordinato per ID serve alla diagnostica, non a stimare quote o precisione.
TEST non rappresenta l'intero settore; PROD serve ai risultati finali dopo la verifica.
Questi controlli non modificano raw_sector né i risultati dell'analisi principale.


In [ ]:
# Nuova cella subito dopo la cella 18.
# La connessione patstat/db e il tuo raw_sector restano quelli esistenti.
test_ids = sorted(raw_sector.appln_id.dropna().astype(int).unique())[:200]
if not test_ids:
    raise ValueError('Nessuna domanda: verificare prima la cella 18')
TEST_IDS_SQL = ', '.join(str(int(x)) for x in test_ids)

def run_check(sql):
    return pd.DataFrame(patstat.sql_query(sql, use_legacy_sql=False))


## T1 - Unita di conteggio

Famiglie <= domande; i collegamenti e le persone sono unita diverse dai nomi armonizzati.


In [ ]:
if RUN_SQL_CHECKS:
    sql = f"""
    SELECT COUNT(*) AS applicant_links,
           COUNT(DISTINCT a.appln_id) AS applications,
           COUNT(DISTINCT a.docdb_family_id) AS families,
           COUNT(DISTINCT p.person_id) AS persons
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON pa.appln_id = a.appln_id
    JOIN tls206_person p ON p.person_id = pa.person_id
    WHERE a.appln_id IN ({TEST_IDS_SQL}) AND pa.applt_seq_nr > 0
    """
    display(run_check(sql))

else:
    print("Controllo SQL disattivato: RUN_SQL_CHECKS=False")


## T2 - Criteri CPC

Leggere le famiglie con charging_core=0 prima di valutarne la rilevanza EV.


In [ ]:
if RUN_SQL_CHECKS:
    sql = f"""
    WITH flags AS (
     SELECT a.docdb_family_id,
       MAX(CASE WHEN REPLACE(c.cpc_class_symbol, ' ', '')
         LIKE 'B60L53/%' THEN 1 ELSE 0 END) AS charging_core,
       MAX(CASE WHEN REPLACE(c.cpc_class_symbol, ' ', '')
         LIKE 'H02J7/00%' THEN 1 ELSE 0 END) AS general_charging,
       MAX(CASE WHEN REPLACE(c.cpc_class_symbol, ' ', '')
         LIKE 'Y02T10/70%' THEN 1 ELSE 0 END) AS ev_storage
     FROM tls201_appln a
     LEFT JOIN tls224_appln_cpc c ON c.appln_id = a.appln_id
     WHERE a.appln_id IN ({TEST_IDS_SQL})
     GROUP BY a.docdb_family_id
    )
    SELECT * FROM flags ORDER BY docdb_family_id
    """
    display(run_check(sql))

else:
    print("Controllo SQL disattivato: RUN_SQL_CHECKS=False")


## T3 - Richiedenti sugli altri membri

Le persone aggiuntive del campione non sono necessariamente assenti dal raw_sector completo: il secondo confronto tiene conto di questo.


In [ ]:
if RUN_SQL_CHECKS:
    sql = f"""
    WITH selected_families AS (
     SELECT DISTINCT docdb_family_id FROM tls201_appln
     WHERE appln_id IN ({TEST_IDS_SQL}) AND docdb_family_id > 0
    ), observed AS (
     SELECT DISTINCT a.docdb_family_id, pa.person_id
     FROM tls201_appln a
     JOIN tls207_pers_appln pa ON pa.appln_id = a.appln_id
     WHERE a.appln_id IN ({TEST_IDS_SQL}) AND pa.applt_seq_nr > 0
    ), expanded AS (
     SELECT DISTINCT a.docdb_family_id, pa.person_id, p.psn_name
     FROM tls201_appln a
     JOIN selected_families f ON f.docdb_family_id = a.docdb_family_id
     JOIN tls207_pers_appln pa ON pa.appln_id = a.appln_id
     JOIN tls206_person p ON p.person_id = pa.person_id
     WHERE pa.applt_seq_nr > 0 AND a.ipr_type = 'PI'
       AND a.appln_id < 900000000
    )
    SELECT e.* FROM expanded e
    LEFT JOIN observed o ON o.docdb_family_id = e.docdb_family_id
                       AND o.person_id = e.person_id
    WHERE o.person_id IS NULL
    ORDER BY e.docdb_family_id, e.person_id
    LIMIT 50
    """
    missing_sample = run_check(sql)
    display(missing_sample)

    if missing_sample.empty:
        print("Nessuna persona aggiuntiva nel test limitato")
    else:
        observed_all = raw_sector[["docdb_family_id", "person_id"]].drop_duplicates()
        audit = missing_sample.merge(observed_all,
            on=["docdb_family_id", "person_id"], how="left", indicator=True)
        display(audit[audit["_merge"].eq("left_only")])

else:
    print("Controllo SQL disattivato: RUN_SQL_CHECKS=False")


## T4 - Titoli da leggere

Un titolo inglese nullo non dimostra irrilevanza; controllare anche abstract e altri membri.


In [ ]:
if RUN_SQL_CHECKS:
    sql = f"""
    SELECT a.appln_id, a.docdb_family_id, a.earliest_filing_year,
           t.appln_title, t.appln_title_lg
    FROM tls201_appln a
    LEFT JOIN tls202_appln_title t
     ON t.appln_id = a.appln_id AND t.appln_title_lg = 'en'
    WHERE a.appln_id IN ({TEST_IDS_SQL})
    ORDER BY a.docdb_family_id, a.appln_id
    LIMIT 50
    """
    display(run_check(sql))

else:
    print("Controllo SQL disattivato: RUN_SQL_CHECKS=False")


## T5 - Varianti dei nomi

Usare questo elenco per controllare gli alias; il nome PATSTAT non dimostra il controllo societario.


In [ ]:
if RUN_SQL_CHECKS:
    sql = f"""
    SELECT p.psn_name, p.person_name,
           COUNT(DISTINCT a.docdb_family_id) AS families
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON pa.appln_id = a.appln_id
    JOIN tls206_person p ON p.person_id = pa.person_id
    WHERE a.appln_id IN ({TEST_IDS_SQL}) AND pa.applt_seq_nr > 0
    GROUP BY p.psn_name, p.person_name
    ORDER BY families DESC, p.psn_name, p.person_name
    LIMIT 100
    """
    display(run_check(sql))

else:
    print("Controllo SQL disattivato: RUN_SQL_CHECKS=False")


## T6 - Anni discordanti

Esito desiderato: nessuna riga. Il controllo copre solo gli ID del campione.


In [ ]:
if RUN_SQL_CHECKS:
    sql = f"""
    SELECT docdb_family_id,
           MIN(earliest_filing_year) AS first_year,
           MAX(earliest_filing_year) AS last_year,
           COUNT(DISTINCT earliest_filing_year) AS different_years
    FROM tls201_appln
    WHERE appln_id IN ({TEST_IDS_SQL})
    GROUP BY docdb_family_id
    HAVING COUNT(DISTINCT earliest_filing_year) > 1
    """
    display(run_check(sql))

else:
    print("Controllo SQL disattivato: RUN_SQL_CHECKS=False")


# Analisi SDG - identificazione con SQL, CPC e testo

**Questa sezione produce gia' un'analisi esplorativa senza compilare prima un CSV.**
Esegui il notebook su TIP: RUN_SDG_ANALYSIS e' attivo e il default rimane TEST.
Per l'analisi definitiva usa PROD dopo aver verificato query e copertura.
Le proposte sono calcolate su tutte le domande del tuo settore osservato, a blocchi;
SDG_MAX_APPLN puo' limitare il lavoro per una prima prova. Non usa modelli ML.

## Come viene proposto un SDG

| SDG | Segnale operativo esplorativo | Perche' puo' essere pertinente |
|---|---|---|
| 7 - Energia | Y02T10/92; oppure B60L53/ con riferimenti a efficienza energetica, solare o rinnovabili | Efficienza (7.3) o energia rinnovabile (7.2) |
| 9 - Infrastrutture | Y02T90/12; oppure B60L53/ con stazioni/infrastrutture di ricarica nel testo | Infrastrutture, da valutare rispetto al target 9.1 |
| 11 - Citta' sostenibili | B60L53/ con electric bus o public transport nel testo | Possibile relazione col trasporto pubblico, target 11.2 |
| 13 - Clima | Tag Y02T di ricarica: 90/12, 10/7072, 10/92 | Pertinenza alla mitigazione climatica; il target preciso resta da verificare |

Le regole sono proposte per il tuo progetto: **non una mappatura ufficiale CPC-SDG**.
Il codice richiede il CPC e il testo sulla stessa domanda per le regole testuali.
Usa titolo/abstract inglese: testi mancanti o in altre lingue possono far perdere corrispondenze.
Una menzione puo' riguardare contesto, negazioni o stato dell'arte: leggere sempre il brevetto.
La presenza di un segnale non prova impatto positivo, efficienza reale o emissioni evitate.
I restanti 13 SDG non sono ricercati da queste regole EV; non considerarli automaticamente assenti.

## Cosa guardare

1. sdg_candidates: ID della domanda e famiglia, SDG proposto, regola, CPC e testo.
2. sdg_auto_summary: famiglie distinte per SDG e quota sull'intero settore osservato.
3. Tabelle per anno e richiedente: sempre marcate come proposte non validate.
4. sdg_review_template: una riga per famiglia-SDG, con decision='uncertain'.

Per confermare una proposta leggi la pubblicazione, controlla il target ONU e cambia
la decisione in accepted o rejected. Risolvi le decisioni dubbie; puoi aggiungere
associazioni non trovate dalle regole se sono documentate. Con EXPORT_FILES=True il
modello viene esportato in OUT_DIR/sdg_review_template.csv. Copialo dopo la revisione
come sdg_review.csv accanto al notebook, oppure modifica review_path nella cella seguente.
Il file di revisione non viene sovrascritto automaticamente.

La sezione successiva mantiene distinti **risultati proposti** e **associazioni confermate**.
Non e' stata eseguita una validazione del classificatore su un campione umano annotato:
non sono quindi disponibili precision, recall o probabilita' di appartenenza affidabili.


In [ ]:
# Regole esplorative EV: associazioni proposte, non classificazione ufficiale.
# title/abstract: solo inglese. Una regola testuale puo' cogliere anche contesto,
# negazioni o tecnologie citate come stato dell'arte: leggere sempre l'evidenza.
SDG_RULES_VERSION = "ev-sql-cpc-text-v1"
SDG_RULES_SQL = "SELECT 'E7_eff_cpc' AS rule_id, 7 AS sdg, '7.3' AS target_hint, 'Y02T10/92' AS cpc_prefix, '' AS text_pattern, 'Tag di ricarica/scarica energeticamente efficiente dei veicoli' AS reason\nUNION ALL\nSELECT 'E7_eff_text', 7, '7.3', 'B60L53/', '%energy efficien%', 'Ricarica EV e riferimento testuale a efficienza energetica'\nUNION ALL\nSELECT 'E7_solar', 7, '7.2', 'B60L53/', '%solar%', 'Ricarica EV e riferimento testuale a energia solare'\nUNION ALL\nSELECT 'E7_pv', 7, '7.2', 'B60L53/', '%photovoltaic%', 'Ricarica EV e riferimento testuale a fotovoltaico'\nUNION ALL\nSELECT 'E7_renewable', 7, '7.2', 'B60L53/', '%renewable energy%', 'Ricarica EV e riferimento testuale a energia rinnovabile'\nUNION ALL\nSELECT 'I9_station_cpc', 9, '9.1', 'Y02T90/12', '', 'Tag di stazioni di ricarica: infrastruttura candidata da valutare'\nUNION ALL\nSELECT 'I9_station_text', 9, '9.1', 'B60L53/', '%charging station%', 'Ricarica EV e riferimento testuale a stazioni di ricarica'\nUNION ALL\nSELECT 'I9_infra_text', 9, '9.1', 'B60L53/', '%charging infrastructure%', 'Ricarica EV e riferimento testuale a infrastrutture'\nUNION ALL\nSELECT 'U11_bus', 11, '11.2', 'B60L53/', '%electric bus%', 'Ricarica EV e riferimento testuale ad autobus elettrici'\nUNION ALL\nSELECT 'U11_public', 11, '11.2', 'B60L53/', '%public transport%', 'Ricarica EV e riferimento testuale a trasporto pubblico'\nUNION ALL\nSELECT 'C13_station', 13, 'da verificare', 'Y02T90/12', '', 'Tag climatico Y02T per stazioni di ricarica'\nUNION ALL\nSELECT 'C13_charge', 13, 'da verificare', 'Y02T10/7072', '', 'Tag climatico Y02T per sistemi di ricarica specifici EV'\nUNION ALL\nSELECT 'C13_eff', 13, 'da verificare', 'Y02T10/92', '', 'Tag climatico Y02T per ricarica/scarica efficiente dei veicoli'"

def sdg_query(ids_sql):
    return f"""
WITH sdg_rules AS ({SDG_RULES_SQL})
SELECT a.appln_id, a.docdb_family_id, r.sdg, r.target_hint,
       r.rule_id, r.reason,
       MIN(REPLACE(c.cpc_class_symbol, ' ', '')) AS cpc_evidence,
       MIN(t.appln_title) AS patent_title,
       MIN(CASE WHEN r.text_pattern <> '' THEN
           SUBSTR(CONCAT(COALESCE(t.appln_title, ''), ' ',
                         COALESCE(b.appln_abstract, '')), 1, 600)
           ELSE NULL END) AS text_excerpt
FROM tls201_appln a
JOIN tls224_appln_cpc c ON c.appln_id = a.appln_id
LEFT JOIN tls202_appln_title t
 ON t.appln_id = a.appln_id AND t.appln_title_lg = 'en'
LEFT JOIN tls203_appln_abstr b
 ON b.appln_id = a.appln_id AND b.appln_abstract_lg = 'en'
JOIN sdg_rules r ON REPLACE(c.cpc_class_symbol, ' ', '')
                   LIKE CONCAT(r.cpc_prefix, '%')
WHERE a.appln_id IN ({ids_sql})
 AND (r.text_pattern = '' OR
      LOWER(CONCAT(COALESCE(t.appln_title, ''), ' ',
                   COALESCE(b.appln_abstract, ''))) LIKE r.text_pattern)
GROUP BY a.appln_id, a.docdb_family_id, r.sdg, r.target_hint,
         r.rule_id, r.reason
ORDER BY a.docdb_family_id, r.sdg, a.appln_id, r.rule_id
"""


In [ ]:
# Tutte le domande del settore osservato, in blocchi piccoli.
# SDG_MAX_APPLN=None evita il limite di 200 ID usato dai test diagnostici.
candidate_columns = ["appln_id", "docdb_family_id", "sdg", "target_hint",
                     "rule_id", "reason", "cpc_evidence", "patent_title",
                     "text_excerpt", "status", "source", "method_version"]
sdg_candidates = pd.DataFrame(columns=candidate_columns)
sdg_scope_families = set()
if RUN_SDG_ANALYSIS:
    sdg_ids = sorted(raw_sector.appln_id.dropna().astype(int).unique())
    if SDG_MAX_APPLN is not None:
        sdg_ids = sdg_ids[:int(SDG_MAX_APPLN)]
    sdg_scope_families = set(raw_sector.loc[
        raw_sector.appln_id.isin(sdg_ids), "docdb_family_id"].unique())
    sdg_parts = []
    for start in range(0, len(sdg_ids), SDG_BATCH_SIZE):
        ids_sql = ", ".join(str(int(x)) for x in
                            sdg_ids[start:start + SDG_BATCH_SIZE])
        part = run_check(sdg_query(ids_sql))
        if not part.empty:
            sdg_parts.append(part)
    if sdg_parts:
        sdg_candidates = pd.concat(sdg_parts, ignore_index=True)
        sdg_candidates = sdg_candidates.drop_duplicates(
            ["appln_id", "docdb_family_id", "sdg", "rule_id"])
        sdg_candidates["status"] = "proposed_unvalidated"
        sdg_candidates["source"] = sdg_candidates.appln_id.map(
            lambda x: "PATSTAT appln_id=" + str(int(x)))
        sdg_candidates["method_version"] = SDG_RULES_VERSION
        sdg_candidates = sdg_candidates.reindex(columns=candidate_columns)
    print("Associazioni PROPOSTE: da revisionare, non SDG confermati")
    print("Domande esaminate:", len(sdg_ids),
          "| famiglie nel perimetro:", len(sdg_scope_families))
    display(sdg_candidates.head(100))
else:
    print("Analisi SDG disattivata: RUN_SDG_ANALYSIS=False")


In [ ]:
# Risultati automatici esplorativi, separati da family_sdg confermato.
sdg_auto_family = sdg_candidates[["docdb_family_id", "sdg"]].drop_duplicates()
sdg_auto_summary = pd.DataFrame()
sdg_review_template = pd.DataFrame(columns=[
    "docdb_family_id", "sdg", "decision", "target", "evidence", "source"])
sdg_auto_coverage = {}
if RUN_SDG_ANALYSIS:
    counts = sdg_auto_family.groupby("sdg").docdb_family_id.nunique()
    sdg_auto_summary = counts.reindex([7, 9, 11, 13], fill_value=0)
    sdg_auto_summary = sdg_auto_summary.rename("families").reset_index()
    total = family_applicants.docdb_family_id.nunique()
    with_sdg = sdg_auto_family.docdb_family_id.nunique()
    sdg_auto_coverage = {
        "observed_families": int(total),
        "families_in_SDG_query_scope": len(sdg_scope_families),
        "with_proposed_SDG": int(with_sdg),
        "without_proposed_SDG": int(total - with_sdg),
        "scope_is_full_observed_sector":
            sdg_scope_families == set(family_applicants.docdb_family_id),
    }
    sdg_auto_summary["pct_all_observed"] = (
        100 * sdg_auto_summary.families / total if total else np.nan)
    sdg_auto_summary["status"] = "proposed_unvalidated"
    display(sdg_auto_coverage)
    display(sdg_auto_summary)
    if not sdg_auto_family.empty:
        # Una famiglia conta una sola volta per ciascun SDG.
        auto_year = sdg_auto_family.merge(
            family_years, on="docdb_family_id", validate="many_to_one")
        sdg_auto_trend = auto_year.groupby(
            ["earliest_filing_year", "sdg"]).docdb_family_id.nunique()
        display(sdg_auto_trend.rename("proposed_families").unstack(fill_value=0))
        auto_links = family_applicants.merge(
            sdg_auto_family, on="docdb_family_id", validate="many_to_many")
        sdg_auto_applicants = auto_links.groupby(
            ["clean_name", "sdg"]).fractional_credit.sum()
        display(sdg_auto_applicants.rename("proposed_fractional_families")
                .unstack(fill_value=0))
        evidence_rows = sdg_candidates.copy()
        evidence_rows["review_evidence"] = (
            evidence_rows.rule_id + ": " + evidence_rows.reason
            + " | CPC=" + evidence_rows.cpc_evidence.fillna("")
            + " | testo=" + evidence_rows.text_excerpt.fillna("solo segnale CPC"))
        def distinct_text(values):
            return " | ".join(sorted(set(values.dropna().astype(str))))
        sdg_review_template = evidence_rows.groupby(
            ["docdb_family_id", "sdg"], as_index=False).agg(
                target=("target_hint", distinct_text),
                evidence=("review_evidence", distinct_text),
                source=("source", distinct_text))
        sdg_review_template["decision"] = "uncertain"
        sdg_review_template = sdg_review_template[[
            "docdb_family_id", "sdg", "decision", "target", "evidence", "source"]]
        display(sdg_review_template.head(30))
    if EXPORT_FILES:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        sdg_candidates.to_csv(OUT_DIR / "sdg_proposte_evidenze.csv", index=False)
        sdg_auto_summary.to_csv(OUT_DIR / "sdg_proposte_riepilogo.csv", index=False)
        # Mai sovrascrivere il file delle decisioni del revisore.
        sdg_review_template.to_csv(OUT_DIR / "sdg_review_template.csv", index=False)
    print("Senza proposta non significa non sostenibile. Quote sovrapposte: somma anche >100%.")


In [ ]:
family_sdg = pd.DataFrame(columns=["docdb_family_id", "sdg"])
SDG_ROWS_SQL = None
review_path = Path("sdg_review.csv")
if RUN_SDG_ANALYSIS and review_path.is_file():
    # Dopo la revisione manuale, leggere una riga per famiglia/SDG.
    # CSV: docdb_family_id,sdg,decision,target,evidence,source
    review = pd.read_csv(review_path)
    required = ['target', 'evidence', 'source']
    if review[required].fillna('').astype(str).apply(
        lambda col: col.str.strip().eq('')).any().any():
        raise ValueError('Ogni decisione richiede target, evidenza e fonte')
    if not review.decision.isin(['accepted','rejected','uncertain']).all():
        raise ValueError('Decisione SDG non riconosciuta')
    for col in ['docdb_family_id', 'sdg']:
        values = pd.to_numeric(review[col], errors='raise')
        if values.isna().any() or values.mod(1).ne(0).any():
            raise ValueError('Identificativi non interi')
        review[col] = values.astype('int64')
    if not review.sdg.between(1,17).all():
        raise ValueError('SDG fuori intervallo')
    if not review.docdb_family_id.isin(
        family_applicants.docdb_family_id).all():
        raise ValueError('Famiglia estranea alla tua analisi')
    if review.duplicated(['docdb_family_id','sdg']).any():
        raise ValueError('Risolvi le decisioni duplicate')
    family_sdg = review.loc[review.decision.eq('accepted'),
                           ['docdb_family_id','sdg']]
    SDG_ROWS_SQL = ' UNION ALL '.join(
        f'SELECT {int(f)} AS docdb_family_id, {int(s)} AS sdg'
        for f,s in family_sdg.itertuples(index=False, name=None))

    if family_sdg.empty:
        print("Nessuna associazione accettata: i conteggi SDG restano sospesi")
elif RUN_SDG_ANALYSIS:
    print("Prepara sdg_review.csv con decisioni documentate prima dei conteggi SDG")
else:
    print("Revisione SDG disattivata")


In [ ]:
if RUN_SDG_ANALYSIS and not family_sdg.empty:
    # La CTE contiene solo associazioni accettate, non i candidati.
    all_families = family_applicants.docdb_family_id.nunique()
    sql = f"""
    WITH reviewed_sdg AS ({SDG_ROWS_SQL})
    SELECT sdg, COUNT(DISTINCT docdb_family_id) AS families,
           100.0 * COUNT(DISTINCT docdb_family_id) / {int(all_families)}
             AS pct_all_observed
    FROM reviewed_sdg
    GROUP BY sdg ORDER BY sdg
    """
    display(run_check(sql))
    classified = family_sdg.docdb_family_id.nunique()
    print('Senza SDG accettato:', all_families - classified)

    # Piccola integrazione pandas per riutilizzare i tuoi crediti esistenti.
    sdg_links = family_applicants.merge(family_sdg, on='docdb_family_id')
    display(sdg_links.groupby(['clean_name','sdg'])
            .fractional_credit.sum().rename('fractional_families'))

else:
    print("Conteggi SDG non eseguiti: attivare la sezione e fornire associazioni accettate")


In [ ]:
if RUN_SDG_ANALYSIS and not family_sdg.empty:
    # Stessa famiglia/SDG contata una sola volta; anno dalla tua estrazione.
    year_rows = family_years[family_years.docdb_family_id.isin(
        family_sdg.docdb_family_id)]
    YEAR_ROWS_SQL = ' UNION ALL '.join(
        f'SELECT {int(f)} AS docdb_family_id, {int(y)} AS family_year'
        for f,y in year_rows.itertuples(index=False, name=None))
    sql = f"""
    WITH reviewed_sdg AS ({SDG_ROWS_SQL}),
         family_years AS ({YEAR_ROWS_SQL})
    SELECT y.family_year, s.sdg,
           COUNT(DISTINCT s.docdb_family_id) AS families
    FROM reviewed_sdg s
    JOIN family_years y ON y.docdb_family_id = s.docdb_family_id
    GROUP BY y.family_year, s.sdg
    ORDER BY y.family_year, s.sdg
    """
    display(run_check(sql))

else:
    print("Conteggi SDG non eseguiti: attivare la sezione e fornire associazioni accettate")


## Interpretazione dei risultati SDG

Ogni famiglia puo' avere piu' SDG: le quote possono sommare a piu' del 100%.
I crediti frazionari per richiedente restano quelli dell'analisi originale, separatamente per ogni SDG.
Se limiti SDG_MAX_APPLN, i candidati non rappresentano l'intero settore. Per risultati settoriali bisogna
estendere la revisione e dichiararne copertura e incertezze. Il mapping non misura CO2 evitata
ne' il raggiungimento effettivo dei target ONU.

Fonti di riferimento: [WIPO 2024](https://www.wipo.int/edocs/pubdocs/en/wipo-pub-rn2024-18-en-mapping-innovations.pdf),
[target ONU](https://sdgs.un.org/goals), [CPC Y02T](https://www.uspto.gov/web/patents/classification/cpc/html/cpc-Y02T.html).


# PEST - traccia da completare con fonti

Questa sezione completa la quarta domanda dello Step 2 senza introdurre nuovi algoritmi.
Le righe seguenti sono domande di ricerca, non risultati gia' verificati.

| Fattore | Domanda per EV charging | Fonte e data | Effetto atteso / evidenza | Limiti |
|---|---|---|---|---|
| Politico | Quali incentivi o obblighi influenzano infrastrutture e domanda? | Da documentare | Da analizzare | Da dichiarare |
| Economico | Come incidono costo dell'energia e investimenti? | Da documentare | Da analizzare | Da dichiarare |
| Sociale | Quali ostacoli di accessibilita' e utilizzo emergono? | Da documentare | Da analizzare | Da dichiarare |
| Tecnologico | Quali standard e soluzioni cambiano interoperabilita' ed efficienza? | Da documentare | Da analizzare | Da dichiarare |

Gli entranti restano prime osservazioni nel settore, non nuove imprese o primi depositanti assoluti.
La concentrazione riguarda brevetti, non quote di mercato. Verificare tipo e paese degli attori
prima di descriverli tutti come aziende. Queste variabili sono gia' disponibili in raw_sector,
ma non sono trasformate automaticamente in sede legale o controllo societario.


In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import json
import sys

package_versions = {}
for package in ("numpy", "pandas", "scipy", "SQLAlchemy", "cleanco"):
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        package_versions[package] = "versione non disponibile"
run_metadata = {
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0], "packages": package_versions,
    "environment": ENV, "years": [YEAR_MIN, LAST_COMPLETE_YEAR],
    "sector_cpc": list(SECTOR_CPC_PREFIXES),
    "keywords": list(SECTOR_KEYWORDS),
    "subfield_cpc": list(SUBFIELD_CPC_PREFIXES),
    "name_method": NAME_METHOD, "verified_aliases": VERIFIED_ALIASES,
    "families": int(family_applicants.docdb_family_id.nunique()),
    "rta_thresholds": [MIN_RTA_FAMILIES, MIN_RTA_PORTFOLIO],
    "sql_checks_enabled": RUN_SQL_CHECKS, "sdg_enabled": RUN_SDG_ANALYSIS,
    "sdg_rules_version": SDG_RULES_VERSION, "sdg_max_appln": SDG_MAX_APPLN,
}
display(run_metadata)
if EXPORT_FILES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "run_metadata.json").write_text(
        json.dumps(run_metadata, indent=2), encoding="utf-8")
# Annotare a parte anche l'edizione PATSTAT mostrata da TIP.
